# sRPE vs TRIMP: do hybrid sessions record less heart-rate load than runs?

**Question.** For the same perceived effort (sRPE), does a hybrid session ("Workout" in intervals.icu) record the same HR-based load (TRIMP) as a run?

**Why it matters.** If heart-rate load under-counts hybrid sessions, weekly load, A:C and monotony will all be biased for hybrid athletes.

**Method (6 steps).**
1. Load and filter the data
2. Plot sRPE against TRIMP for both groups
3. Fit a straight line to runs only
4. Score each hybrid session against that line
5. Test whether the hybrid line differs from the run line
6. State the conclusion and its limits

**Definitions.** sRPE = RPE x session duration (minutes). TRIMP = intervals.icu `load` (HR-based).

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.formula.api as smf

from hll.config import PROCESSED_DIR

# Settings: change these, not the code below
DAYS = 180                                  # how far back to look
MIN_MINS = 5                                # ignore sessions shorter than this (junk recordings)
RUN_TYPES = ["Run", "VirtualRun", "TrailRun"]
HYBRID_TYPES = ["Workout"]                  # what intervals.icu calls my hybrid sessions
COLORS = {"run": "#9aa0a6", "hybrid": "#0f9d8a"}

## Step 1: Load and filter

We keep only runs and hybrid sessions, then build the two variables we need (`srpe` and `trimp`).
We only keep sessions where both RPE and HR load exist, because the comparison needs both.

In [ ]:
act = pd.read_csv(PROCESSED_DIR / "activities.csv", parse_dates=["start_date"])

# Keep the last DAYS days
cutoff = act["start_date"].max() - pd.Timedelta(days=DAYS)
act = act[act["start_date"] >= cutoff]

# Keep runs and hybrid sessions only, and label each one
df = act[act["type"].isin(RUN_TYPES + HYBRID_TYPES)].copy()
df["group"] = np.where(df["type"].isin(HYBRID_TYPES), "hybrid", "run")

# Build the two variables
df["duration_min"] = df["elapsed_s"] / 60       # elapsed time: the whole session, as you rate it
df["srpe"] = df["rpe"] * df["duration_min"]     # sRPE = RPE x minutes
df["trimp"] = df["load"]                        # HR-based load

# Keep only usable sessions
keep = (
    (df["moving_s"] / 60 >= MIN_MINS)           # long enough to be a real session
    & df["rpe"].notna()                         # RPE was logged
    & df["hr_load"].notna()                     # HR load exists
    & (df["trimp"] > 0)
)
df = df[keep].copy()

df.groupby("group").size().rename("usable sessions")

## Step 2: Scatter plot

Each dot is one session. The lines are straight-line fits per group. **r** is the correlation:
1 means the points sit exactly on a line, 0 means no linear relationship.

In [ ]:
fig = px.scatter(
    df, x="srpe", y="trimp", color="group", trendline="ols",
    color_discrete_map=COLORS, hover_data=["start_date", "type", "rpe", "duration_min"],
    labels={"srpe": "sRPE (RPE x minutes)", "trimp": "TRIMP (HR-based load)"},
)
fig.show()

# Correlation and sample size per group
summary = pd.DataFrame(
    {g: {"n": len(x), "r": x["srpe"].corr(x["trimp"])} for g, x in df.groupby("group")}
).T
summary.round(2)

## Step 3: Fit a line to runs only

Runs are our reference. We fit `trimp = intercept + slope x srpe` using only runs, then ask how hybrid
sessions compare with it. (This is the same as `lm(trimp ~ srpe)` in R.)

- **slope**: extra TRIMP per extra unit of sRPE
- **intercept**: TRIMP at zero sRPE (should be near 0; a large value means something else is going on)
- **CI**: 95% confidence interval, the range of values the data are compatible with

In [ ]:
runs = df[df["group"] == "run"]
hyb = df[df["group"] == "hybrid"].copy()

run_model = smf.ols("trimp ~ srpe", data=runs).fit()

print(f"n runs = {len(runs)},  R-squared = {run_model.rsquared:.2f}")
pd.concat([run_model.params, run_model.conf_int()], axis=1).set_axis(
    ["estimate", "ci_low", "ci_high"], axis=1
).round(3)

**Check the model.** Residuals are the gaps between each run's actual TRIMP and the line. If the line is a good
description, the dots should scatter evenly around 0 with no curve or funnel shape.

In [ ]:
fig_resid = px.scatter(
    x=run_model.fittedvalues, y=run_model.resid,
    labels={"x": "Fitted TRIMP", "y": "Residual (actual - fitted)"},
    title="Run model: residuals vs fitted",
)
fig_resid.add_hline(y=0, line_dash="dash")
fig_resid.show()

## Step 4: Score hybrid sessions against the run line

For each hybrid session we ask: *what TRIMP would a run with the same sRPE have recorded?* Then
`ratio = actual TRIMP / predicted TRIMP`.

- ratio = 1: same HR load as a run of equal effort
- ratio < 1: HR load is lower than a run of equal effort (what my hypothesis predicts)
- ratio > 1: HR load is higher

In [ ]:
hyb["pred"] = run_model.predict(hyb)
hyb["ratio"] = hyb["trimp"] / hyb["pred"]

fig_ratio = px.scatter(
    hyb, x="srpe", y="ratio", hover_data=["start_date", "rpe", "duration_min", "trimp"],
    labels={"srpe": "sRPE (RPE x minutes)", "ratio": "Actual / run-predicted TRIMP"},
    color_discrete_sequence=[COLORS["hybrid"]],
)
fig_ratio.add_hline(y=1, line_dash="dash")
fig_ratio.show()

print(f"Median ratio: {hyb['ratio'].median():.2f}  (n = {len(hyb)} hybrid sessions)")
hyb.sort_values("ratio")[["start_date", "rpe", "duration_min", "srpe", "trimp", "pred", "ratio"]].round(
    {"duration_min": 1, "srpe": 0, "trimp": 1, "pred": 1, "ratio": 2}
)

**How uncertain is that median?** We have few sessions, so we resample. Each round: draw runs with
replacement, draw hybrid sessions with replacement, refit the run line, recompute the median ratio.
Repeating 2000 times shows how much the answer moves around by chance (the middle 95% is the interval).

In [ ]:
rng = np.random.default_rng(42)
medians = []
for _ in range(2000):
    r = runs.sample(len(runs), replace=True, random_state=int(rng.integers(1e9)))
    h = hyb.sample(len(hyb), replace=True, random_state=int(rng.integers(1e9)))
    slope, intercept = np.polyfit(r["srpe"], r["trimp"], 1)
    medians.append(np.median(h["trimp"] / (intercept + slope * h["srpe"])))

lo, hi = np.percentile(medians, [2.5, 97.5])
print(f"Hybrid sessions recorded {np.median(hyb['ratio']):.0%} of the TRIMP that runs of the same sRPE did "
      f"(95% interval {lo:.0%} to {hi:.0%}), n = {len(hyb)} hybrid vs {len(runs)} runs")

## Step 5: Do the hybrid and run lines differ?

Step 4 asked about one number. This step asks about the whole line. We fit both groups in one model:

`trimp = a + b x srpe + c x is_hybrid + d x (srpe x is_hybrid)`

- **c** (`is_hybrid`): how much higher or lower the hybrid line starts
- **d** (`srpe:is_hybrid`): how much steeper or flatter the hybrid line is

If the 95% interval for both c and d includes 0, the data cannot tell the two lines apart.

In [ ]:
df["is_hybrid"] = (df["group"] == "hybrid").astype(int)
both_model = smf.ols("trimp ~ srpe * is_hybrid", data=df).fit()

pd.concat([both_model.params, both_model.conf_int(), both_model.pvalues], axis=1).set_axis(
    ["estimate", "ci_low", "ci_high", "p_value"], axis=1
).round(3)

## Step 6: Conclusion and limits

*Write this in your own words after running the notebook. Fill in the brackets.*

**Result.** Hybrid sessions recorded [X]% of the TRIMP that runs did at the same sRPE (95% interval [low]% to [high]%).
The interaction terms were [c = ... , CI ...; d = ... , CI ...].

**What it means.** [Does the interval include 100%? If yes: no detectable difference at this sample size. If no: in which direction?]

**Why I should be careful.**
- Small sample (n = [..] hybrid, [..] runs), single athlete, so the interval is wide
- "Workout" in intervals.icu mixes session types (station-heavy HYROX work vs mixed circuits)
- Wrist HR can under-read or lag in intermittent work, and I had faulty readings from an arm strap earlier on
- sRPE depends on how I rate effort, and RPE is only logged on some sessions
- Even a real gap would not say *why*: it could be mechanical load that HR misses, or measurement error

**Next steps.**
- Log RPE on every session, plus separate legs and breathing RPE on hybrid sessions
- Flag sessions with bad HR and repeat the analysis without them
- Split station-heavy sessions from other hybrid sessions